# Tome C3, Chapter 12: Circuits and Proof Complexity (Python companion)

Exercises first, solutions at the end. We count circuits as Shannon did, find the smallest circuit of every function of three variables, watch random polynomials imitate $\lor$ gates, measure how badly low-degree polynomials fit parity, pluck sunflowers and check the error bound of Razborov's method, watch DPLL drown in pigeons, verify the Beame–Pitassi swap lemma on all critical assignments, and test a natural property.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools, math, random, time
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
random.seed(12)
rng = np.random.default_rng(12)

## Exercises

**P12.1 (★) Shannon's count.** Using the bound $s(3(n+s+2)^2)^s$ on the number of functions computed by circuits of size at most $s$ (proof of Theorem 12.3), find for $n=4,\dots,20$ the largest $s$ for which this bound is below $2^{2^n}$, and compare it with $2^n/n$.

**P12.2 (★★) All circuits of three variables.** With gates computing arbitrary functions of two bits (and outputs that may be negated for free), compute by exhaustive search the minimum circuit size of each of the $256$ functions of three variables. What are the sizes of majority and of parity, and how many functions need the maximum?

**P12.3 (★★) Random polynomials for $\lor$.** Implement the random polynomial of Lemma 12.5 for an $\lor$ gate of fan-in $k$ over $\mathbb F_3$, and measure its error probability at a fixed input with a single $1$, and over all inputs, for $\ell=1,\dots,8$. Then approximate a random DNF with $s$ terms on $12$ variables and compare the fraction of wrong inputs with $s2^{-\ell}$.

**P12.4 (★★) Parity against low degree.** For $n=4$ and $n=5$, find by exhaustive search the largest number of points of $\{0,1\}^n$ on which a polynomial of degree at most $D$ over $\mathbb F_3$ agrees with parity, and compare with the bound $\sum_{i\le n/2+D}\binom ni$ of Lemma 12.6.

**P12.5 (★★) Plucking sunflowers.** Implement the proof of the sunflower lemma as an algorithm that finds $p$ petals in any family of more than $\ell!(p-1)^\ell$ sets of size at most $\ell$. On random families of subsets of $\{1,\dots,n\}$, pluck, and estimate on random colourings with $k-1$ colours the probability that the core is rainbow while no petal is; compare with $2^{-p}$.

**P12.6 (★) Pigeons drown DPLL.** Run a DPLL solver (Tome C1, Chapter 1) on $\mathrm{PHP}_n$ for $n=1,\dots,7$ and count the leaves of its search tree, which is a tree-like resolution refutation. How fast does it grow?

**P12.7 (★★) The swap lemma.** For $\mathrm{PHP}_4$, draw random clauses $C$, compute $\mathrm{pig}(C)$ by enumerating the $5!$ critical assignments and the monotone clause $C^+$, and check the inequality $|C^+|\ge|\mathrm{pig}(C)|\,(m+1-|\mathrm{pig}(C)|)$ from the proof of Lemma 12.11.

**P12.8 (★★) A natural property.** Let $\Gamma_n$ be the set of functions whose algebraic normal form over $\mathbb F_2$ has degree $n$. Show it is constructive and large, and that it is useful against circuits with fewer than $n-1$ gates $\land$ (gates $\oplus$ and $\lnot$ being free). Then test it on random circuits with few $\land$ gates, and on a small keyed "mixing" circuit in the style of a block cipher: does it separate them from random functions?

## Solutions

### P12.1
We work with logarithms, since the numbers are astronomical.

In [ ]:
def log2_count(n, s):
    return math.log2(s) + s * math.log2(3 * (n + s + 2) ** 2)

print(" n   largest s with count < 2^(2^n)    2^n/n     ratio")
for n in range(4, 21, 2):
    lo, hi = 1, 2 ** n
    while lo < hi:                          # binary search for the threshold
        mid = (lo + hi + 1) // 2
        if log2_count(n, mid) < 2 ** n:
            lo = mid
        else:
            hi = mid - 1
    print(f"{n:2d}   {lo:14d}              {2**n / n:12.0f}   {lo / (2**n / n):.3f}")

The threshold grows like $2^n/n$, with a factor between $0.6$ and $0.7$ that slowly decreases towards $\frac12$: circuits of size $c\,2^n/n$ with $c<\frac12$ are simply too few to cover all functions, whatever they look like. Lupanov's theorem says the truth is a factor $1$; the counting loses the factor $2$ because it counts each circuit many times (gates can be listed in many orders).

### P12.2
A function of three variables is a truth table of $8$ bits, an integer below $256$. We grow sets of computed functions breadth first: a circuit of size $s$ is a set of $s$ new functions, each obtained from two earlier ones by one of the ten binary operations that depend on both arguments.

In [ ]:
n = 3; N = 1 << n; FULL = (1 << N) - 1
xs = [sum(((i >> k) & 1) << i for i in range(N)) for k in range(n)]
ops = [lambda a, b: a & b, lambda a, b: a & ~b & FULL, lambda a, b: ~a & b & FULL, lambda a, b: ~(a | b) & FULL,
       lambda a, b: a | b, lambda a, b: (a | ~b) & FULL, lambda a, b: (~a | b) & FULL, lambda a, b: ~(a & b) & FULL,
       lambda a, b: a ^ b, lambda a, b: ~(a ^ b) & FULL]
t0 = time.perf_counter()
size = {0: 0, FULL: 0}
for x in xs:
    size[x] = 0; size[x ^ FULL] = 0
level = {frozenset(xs)}
s = 0
while len(size) < 256:
    s += 1
    new = set()
    for st in level:
        for a, b in itertools.combinations(st, 2):
            for op in ops:
                f = op(a, b)
                if f in st:
                    continue
                for g in (f, f ^ FULL):
                    size.setdefault(g, s)
                new.add(st | {f})
    level = new
maj = (xs[0] & xs[1]) | (xs[0] & xs[2]) | (xs[1] & xs[2])
par = xs[0] ^ xs[1] ^ xs[2]
print(f"search took {time.perf_counter() - t0:.1f} s")
print("number of functions by minimum size:", dict(sorted(Counter(size.values()).items())))
print("majority:", size[maj], "  parity:", size[par], "  x1 AND x2 AND x3:", size[xs[0] & xs[1] & xs[2]])
hardest = [f for f in size if size[f] == max(size.values())]
print("examples of hardest functions (truth tables):", [format(f, "08b") for f in hardest[:6]])

Every function of three variables has a circuit with at most $4$ gates of this kind, and $24$ functions need exactly $4$; majority is one of them, which confirms that the circuit of Figure 12.1 is optimal, while parity needs only $2$ gates because exclusive or is in the basis. Already for four variables this exhaustive search becomes painful, and for five it is a research computation: finding the minimum circuit of a given truth table is the problem MCSP, which is in $\mathsf{NP}$ but not known to be $\mathsf{NP}$-complete, a question tied to the natural proofs barrier itself.

### P12.3
Values in $\mathbb F_3$ are integers modulo $3$; we evaluate the random polynomial directly on inputs rather than expanding it.

In [ ]:
def or_poly_eval(X, ell, r):
    # X: (num_points, k) array of 0/1 values of the inputs; returns the random polynomial at each point
    k = X.shape[1]
    prod = np.ones(X.shape[0], dtype=np.int64)
    for _ in range(ell):
        S = r.random(k) < 0.5
        sm = X[:, S].sum(axis=1) % 3
        prod = prod * (1 - sm * sm) % 3
    return (1 - prod) % 3

k = 20
one_hot = np.zeros((1, k), dtype=np.int64); one_hot[0, 7] = 1
X = rng.integers(0, 2, size=(4000, k)); X[:10] = 0
truth = X.max(axis=1)
print(" ell   P(wrong) at e_7   fraction wrong over random inputs   2^-ell")
for ell in range(1, 9):
    wrong1 = np.mean([or_poly_eval(one_hot, ell, rng)[0] != 1 for _ in range(2000)])
    wrongX = np.mean(or_poly_eval(X, ell, rng) != truth)
    print(f"{ell:4d}   {wrong1:15.4f}   {wrongX:33.4f}   {2**-ell:.4f}")

In [ ]:
# approximate a random DNF: OR of s ANDs of 3 literals, on 12 variables, evaluated on all 4096 points
nv = 12
pts = np.array(list(itertools.product((0, 1), repeat=nv)), dtype=np.int64)
def dnf_eval(terms, P):
    vals = [np.all([(P[:, abs(l) - 1] == (1 if l > 0 else 0)) for l in t], axis=0) for t in terms]
    return np.any(vals, axis=0).astype(np.int64)
def dnf_poly_eval(terms, P, ell, r):
    # AND gate = NOT OR NOT, literals exact; then the OR gate
    ands = []
    for t in terms:
        lits = np.stack([P[:, abs(l) - 1] if l > 0 else 1 - P[:, abs(l) - 1] for l in t], axis=1)
        ands.append((1 - or_poly_eval(1 - lits, ell, r)) % 3)
    return or_poly_eval(np.stack(ands, axis=1), ell, r)

for s in [10, 40]:
    terms = [tuple(random.choice((-1, 1)) * v for v in random.sample(range(1, nv + 1), 3)) for _ in range(s)]
    truth = dnf_eval(terms, pts)
    row = []
    for ell in [2, 4, 6, 8, 10]:
        bad = np.mean([np.mean(dnf_poly_eval(terms, pts, ell, rng) != truth) for _ in range(3)])
        row.append(f"ell={ell}: {bad:.4f} (bound {min(1, (s + 1) * 2**-ell):.4f})")
    print(f"s = {s}: " + ";  ".join(row))

At an input where exactly one argument of the $\lor$ is $1$, each random subset contains it with probability $\frac12$, so the error is exactly $2^{-\ell}$; at a typical input with many $1$s the error is smaller, about $3^{-\ell}$, since a random subset sum is then nearly uniform modulo $3$. For the DNF the fraction of wrong points stays below the union bound $(s+1)2^{-\ell}$ (one $\lor$ gate and $s$ $\land$ gates), usually far below, because most inputs are easy for most gates. The degree, meanwhile, is $(2\ell)^2$: this trade-off between error and degree is the whole proof of Theorem 12.7.

### P12.4
We list the polynomials of degree at most $D$ through their coefficients on the multilinear monomials, and evaluate them all at once with a matrix product modulo $3$. (Non-multilinear polynomials agree on the cube with multilinear ones of no larger degree, Exercise 12.4.)

In [ ]:
def best_agreement(n, D, chunk=200_000):
    pts = np.array(list(itertools.product((0, 1), repeat=n)), dtype=np.int64)
    mons = [I for d in range(D + 1) for I in itertools.combinations(range(n), d)]
    Mv = np.array([[int(all(p[i] for i in I)) for I in mons] for p in pts], dtype=np.int64)   # points x monomials
    par = pts.sum(axis=1) % 2
    best, total = 0, 3 ** len(mons)
    for start in range(0, total, chunk):
        idx = np.arange(start, min(total, start + chunk))
        coef = (idx[:, None] // 3 ** np.arange(len(mons))) % 3                    # all coefficient vectors
        vals = coef @ Mv.T % 3
        best = max(best, int((vals == par).sum(axis=1).max()))
    return best, len(mons)

t0 = time.perf_counter()
for n, Ds in [(4, [0, 1, 2]), (5, [0, 1])]:
    for D in Ds:
        b, nm = best_agreement(n, D)
        bound = sum(math.comb(n, i) for i in range(n + 1) if i <= n / 2 + D)
        print(f"n = {n}, D = {D}: {3**nm:7d} polynomials, best agreement {b:2d} of {2**n}, Lemma 12.6 bound {bound}")
print(f"({time.perf_counter() - t0:.1f} s)")

A constant agrees with parity on half the points, and degree $1$ gains only a few more. Degree $2$ does much better for $n=4$, but $2$ is already half of $n$: the lemma is about degrees far below $\sqrt n$, where it says that degree $D$ buys at most a fraction about $D/\sqrt n$ of the cube beyond one half. For such tiny $n$ its bound is loose, and for $n=4$, $D=2$ it is trivial. Parity is the function that ``looks at every variable at once'', and low-degree polynomials only see a few at a time.

### P12.5
The proof of the Erdős–Rado lemma is an algorithm: take a maximal family of pairwise disjoint sets; if it has $p$ members, they form a sunflower with empty core; otherwise some element lies in many sets, and we recurse on those sets with this element removed.

In [ ]:
def find_sunflower(F, p):
    # F: list of frozensets (distinct), returns (core, petals) or None
    disjoint, used = [], set()
    for S in F:
        if not (S & used):
            disjoint.append(S); used |= S
    if len(disjoint) >= p:
        return frozenset(), disjoint[:p]
    if not used:
        return None
    x = max(used, key=lambda v: sum(v in S for S in F))
    sub = [S - {x} for S in F if x in S]
    res = find_sunflower(list(dict.fromkeys(sub)), p)
    if res is None:
        return None
    core, petals = res
    return core | {x}, [P | {x} for P in petals]

def pluck(F, p, M):
    F = list(dict.fromkeys(F)); log = []
    while len(F) > M:
        core, petals = find_sunflower(F, p)
        assert all(P & Q == core for P, Q in itertools.combinations(petals, 2))
        F = [S for S in F if S not in petals]
        if core not in F:
            F.append(core)
        log.append((core, petals))
    return F, log

nvert, ell, p, k = 60, 3, 4, 10
M = math.factorial(ell) * (p - 1) ** ell
F = [frozenset(random.sample(range(nvert), ell)) for _ in range(400)]
F2, log = pluck(F, p, M)
print(f"family of {len(set(F))} sets of size {ell}, M = {M}: {len(log)} pluckings, {len(F2)} sets left")
def rainbow(S, col):
    return len({col[v] for v in S}) == len(S)
T = 20000
errs = []
for core, petals in log[:30]:
    bad = 0
    for _ in range(T // 10):
        col = rng.integers(0, k - 1, size=nvert)
        bad += rainbow(core, col) and not any(rainbow(P, col) for P in petals)
    errs.append(bad / (T // 10))
print(f"plucking error over random colourings: mean {np.mean(errs):.4f}, max {max(errs):.4f}; bound 2^-p = {2**-p:.4f}")

The algorithm always finds a sunflower when the family exceeds $M=\ell!(p-1)^\ell$, as the lemma promises, and it often finds one much earlier. The measured plucking errors stay below $2^{-p}$; they are much smaller than the bound because a petal fails to be rainbow with probability far less than $\frac12$ here ($\binom32/9=\frac13$ at most). In Theorem 12.8 the parameters are chosen so that $2^{-p}$ beats the number of pluckings, about $sM^2$, unless the circuit is huge.

### P12.6
A compact DPLL with unit propagation; a run on an unsatisfiable formula is a tree-like resolution refutation whose leaves are the dead ends.

In [ ]:
def php(n):
    var = {(i, j): i * n + j + 1 for i in range(n + 1) for j in range(n)}
    cls = [[var[i, j] for j in range(n)] for i in range(n + 1)]
    cls += [[-var[i, j], -var[i2, j]] for j in range(n) for i, i2 in itertools.combinations(range(n + 1), 2)]
    return cls, len(var)

def dpll_leaves(clauses, assign):
    # returns (satisfiable?, number of leaves of the search tree)
    while True:                                   # unit propagation
        unit = None
        for c in clauses:
            if any(assign.get(abs(l)) == (l > 0) for l in c):
                continue
            free = [l for l in c if abs(l) not in assign]
            if not free:
                return False, 1                   # conflict: a leaf
            if len(free) == 1:
                unit = free[0]; break
        if unit is None:
            break
        assign = {**assign, abs(unit): unit > 0}
    free_vars = {abs(l) for c in clauses for l in c if abs(l) not in assign}
    if not free_vars:
        return True, 1
    v = min(free_vars)
    leaves = 0
    for val in (True, False):
        sat, lv = dpll_leaves(clauses, {**assign, v: val})
        leaves += lv
        if sat:
            return True, leaves
    return False, leaves

t0 = time.perf_counter()
for n in range(1, 8):
    cls, nv = php(n)
    sat, leaves = dpll_leaves(cls, {})
    print(f"PHP_{n}: {nv:2d} variables, {len(cls):4d} clauses, satisfiable={sat}, leaves {leaves:6d}, n! = {math.factorial(n)}")
print(f"({time.perf_counter() - t0:.1f} s)")

The number of leaves grows like $n!$, and even faster: DPLL places pigeons one by one in every possible way before discovering, each time, that the last pigeon has no hole. Tree-like resolution needs this, and Theorem 12.12 shows that even general resolution, which may reuse derived clauses, needs exponential size. Humans refute $\mathrm{PHP}_n$ in one line by counting, which is exactly what resolution cannot express.

### P12.7
Critical assignments of $\mathrm{PHP}_m$ are permutations with one pigeon left out; a clause is a list of literals $\pm(i,j)$.

In [ ]:
m = 4
pigeons, holes = range(m + 1), range(m)
crit = []                                          # (missing pigeon, dict pigeon -> hole)
for i in pigeons:
    others = [a for a in pigeons if a != i]
    for perm in itertools.permutations(holes):
        crit.append((i, dict(zip(others, perm))))
def value(clause, place):
    return any((place.get(a) == b) == pos for pos, a, b in clause)
def plus(clause):
    V = set()
    for pos, a, b in clause:
        if pos:
            V.add((a, b))
        else:
            V |= {(a2, b) for a2 in pigeons if a2 != a}
    return V
checked, by_pig = 0, Counter()
for _ in range(1500):
    w = random.randint(1, 7)
    clause = [(random.random() < 0.5, random.choice(pigeons), random.choice(holes)) for _ in range(w)]
    pig = {i for i, place in crit if not value(clause, place)}
    Cp = plus(clause)
    for i, place in crit:                          # C and C+ agree on critical assignments
        assert value(clause, place) == any(place.get(a) == b for a, b in Cp)
    assert len(Cp) >= len(pig) * (m + 1 - len(pig)), (clause, pig, Cp)
    checked += 1; by_pig[len(pig)] += 1
print(f"{len(crit)} critical assignments; {checked} random clauses checked, all satisfy |C+| >= |pig|(m+1-|pig|)")
print("distribution of |pig(C)|:", dict(sorted(by_pig.items())))

The inequality holds on every clause tried, together with the agreement of $C$ and $C^+$ on critical assignments (Exercise 12.9). Short random clauses tend to be falsified for many pigeons or for none, and those in the middle, the ones the lemma singles out, are always wide. The proof of Theorem 12.12 then only has to show that a refutation with few clauses cannot afford a single wide one.

### P12.8
The algebraic normal form is computed from the truth table by the Möbius transform, in $O(N\log N)$ operations for $N=2^n$: so $\Gamma$ is constructive. Its top coefficient is the parity of the number of $1$s in the truth table, so exactly half of all functions have degree $n$: $\Gamma$ is large. And a circuit with $a$ gates $\land$ and free $\oplus,\lnot$ computes a function of degree at most $a+1$, by induction ($\deg(g\land h)\le\deg g+\deg h$ and $\oplus$ does not raise the degree), so $\Gamma$ is useful against circuits with fewer than $n-1$ gates $\land$.

In [ ]:
def anf_degree(tt, n):
    a = tt.copy(); idx = np.arange(1 << n)
    for k in range(n):                            # Moebius transform over F_2
        m = (idx >> k) & 1 == 1
        a[m] ^= a[idx[m] ^ (1 << k)]
    nz = np.flatnonzero(a)
    return max(bin(i).count("1") for i in nz) if len(nz) else -1

def random_circuit_tt(n, n_and, n_xor, r):
    pts = np.arange(1 << n)
    wires = [((pts >> k) & 1).astype(np.uint8) for k in range(n)]
    gates = ["and"] * n_and + ["xor"] * n_xor
    r.shuffle(gates)
    for g in gates:
        a, b = r.choice(len(wires), size=2, replace=False)
        x, y = wires[a] ^ r.integers(0, 2), wires[b] ^ r.integers(0, 2)    # free negations
        wires.append(x & y if g == "and" else x ^ y)
    return wires[-1]

def mixer_tt(n, rounds, r):
    # keyed rounds: a nonlinear layer (n AND gates) then a linear layer; the key is the random constants
    pts = np.arange(1 << n)
    w = [((pts >> k) & 1).astype(np.uint8) ^ r.integers(0, 2) for k in range(n)]
    for _ in range(rounds):
        w = [w[i] ^ ((1 - w[(i + 1) % n]) & w[(i + 2) % n]) for i in range(n)]
        w = [w[i] ^ w[(i + 3) % n] ^ r.integers(0, 2) for i in range(n)]
    return w[0]

n = 10
t0 = time.perf_counter()
rand_deg = Counter(anf_degree(rng.integers(0, 2, 1 << n).astype(np.uint8), n) for _ in range(200))
print("random functions, ANF degree:", dict(sorted(rand_deg.items())))
for n_and in [5, 8]:
    degs = Counter(anf_degree(random_circuit_tt(n, n_and, 3 * n, rng), n) for _ in range(200))
    print(f"random circuits with {n_and} AND gates: degrees {dict(sorted(degs.items()))}")
for rounds in [2, 3, 4, 6]:
    degs = Counter(anf_degree(mixer_tt(n, rounds, rng), n) for _ in range(200))
    print(f"mixing circuit, {rounds} rounds ({rounds * n} AND gates): fraction of degree {n}: {degs[n] / 200:.3f};  degrees {dict(sorted(degs.items()))}")
print(f"({time.perf_counter() - t0:.1f} s)")

Random functions have degree $n$ half the time and $n-1$ otherwise, while circuits with fewer than $n-1$ gates $\land$ never reach degree $n$: the property is constructive, large and useful against them. But four rounds of the keyed mixing circuit, $40$ gates $\land$ and a few dozen $\oplus$, a tiny circuit, already produce functions of degree $n$ about half the time and $n-1$ otherwise, exactly like random functions; the property cannot tell them apart. (Random wiring is a poor mixer: it leaves the output depending on few variables, which is why the random circuits above have low degree.) A property that small circuits can fake is useless against them, and Razborov and Rudich show that this is unavoidable for every natural property if pseudorandom functions exist: small circuits can imitate random functions so well that no efficient test of the truth table can tell them apart.